In [6]:
# These are dummy. They were changed long before this notebook got near the internet, so they are safe to show. 
credentials = {
        "key": "PKCTYL4MO5SA2QEIZL2TDEJRTA",
        "secret_key": "DKL2eLVYFcxKzMJtDYbuG3zppWjpwHzsTS1DtHVwc9Cz" 
    }

## Utilities

In [30]:
import json
import logging
import re
import requests

from datetime import datetime
from typing import Union

In [1]:
# Used in:
    #get_most_active_stocks
    #get_top_movers
headers = {
    "accept": "application/json",
    "APCA-API-KEY-ID": credentials["key"],
    "APCA-API-SECRET-KEY": credentials["secret_key"]
}

def get_tickers() -> void:
    # get S&P 500 tickers
    # no arguments, returns void, creates json list with tickers
    from bs4 import BeautifulSoup
    
    response = requests.get('https://stockanalysis.com/list/sp-500-stocks/')
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find all <td> elements with the specific class
    td_elements = soup.find_all('td', class_='sym svelte-1ro3niy')
    
    # Extract text from <a> tags inside those <td> elements
    tickers = []
    for td in td_elements:
        a_tag = td.find('a')
        if a_tag and a_tag.text.strip():
            tickers.append(a_tag.text.strip())
            
    # Write those into json file
    with open("tickers.json", "w") as json_file:
        json.dump(tickers, json_file)

# get top active stocks
def get_most_active_stocks(by: str = "volume", top: int = 100) -> str:
    """Calls API and returns pretty formated json string of most active stocks at the moment of last update. 
    Args:
        by: either 'volume' or 'trades'
        top: a number between 1 and 100"""
    
    url = f"https://data.alpaca.markets/v1beta1/screener/stocks/most-actives?by={by}&top={top}"
    response = requests.get(url, headers=headers)
    
    return response.text

# get top market movers
def get_top_movers(top: int = 100) -> str:
    """Calls API and returns pretty formated json string of top market movers, both gainers and losers, at the moment of last update. 
    Unfortunately, it doesn't filter out penny stocks, so we can't look at stocks worth, say, $5 and more.
    Args:
        top: a number between 1 and 50. How many of each gainers and losers to return."""
    
    url = f"https://data.alpaca.markets/v1beta1/screener/stocks/movers?top={top}"
    response = requests.get(url, headers=headers)
    
    return response.text

def validate_time_frame(time_frame: str) -> Union[bool, str]:
    """
    Validate if time_frame satisfies the aggregation format:
    - [1-59]Min or [1-59]T for minutes
    - [1-23]Hour or [1-23]H for hours  
    - 1Day or 1D for days
    - 1Week or 1W for weeks
    - [1,2,3,4,6,12]Month or [1,2,3,4,6,12]M for months
    """
    # ^(\d+) - Capture digits at start
    # (Min|T|Hour|H|Day|D|Week|W|Month|M)$ - Specific suffixes
    match = re.match(r'^(\d+)(Min|T|Hour|H|Day|D|Week|W|Month|M)$', str(time_frame))
    
    if not match:
        return False, "Invalid format. Must be: [number][unit]"
    
    value = int(match.group(1))
    unit = match.group(2)
    
    # Validate based on unit
    if unit in ['Min', 'T']:
        if not (1 <= value <= 59):
            return False, "Minutes must be between 1-59"
            
    elif unit in ['Hour', 'H']:
        if not (1 <= value <= 23):
            return False, "Hours must be between 1-23"
            
    elif unit in ['Day', 'D']:
        if value != 1:
            return False, "Days must be exactly 1"
            
    elif unit in ['Week', 'W']:
        if value != 1:
            return False, "Weeks must be exactly 1"
            
    elif unit in ['Month', 'M']:
        valid_months = [1, 2, 3, 4, 6, 12]
        if value not in valid_months:
            return False, f"Months must be one of {valid_months}"
    
    return True, "Valid time frame"

In [55]:
tickers_to_search = tickers[:10]
tickers_to_search = "%2C".join(tickers)

In [ ]:
# '%2C' is delimiter between tickers

records_limit = 10 # between 1 and 10k May be use limit=aggregation window?
next_page_token = "" 
url = f"https://data.alpaca.markets/v2/stocks/bars?symbols={tickers_to_search}&timeframe=1D&start=2025-11-24&limit={records_limit}&adjustment=raw&feed=sip&sort=asc&page_token={next_page_token}"
response = requests.get(url, headers=headers)
parsed = json.loads(response.text)
# Write into json file here 
json_dict = dict(parsed)
i = 1
while json_dict['next_page_token'] is not None:
    next_page_token = json_dict['next_page_token']
    # update url here
    response = requests.get(url, headers=headers)
    parsed = json.loads(response.text)
    json_dict = dict(parsed)
    i +=1
    # Write into json file here again
print(i)
print(json.dumps(parsed, indent=4))

In [ ]:
# this and above will merge in one function
# args:
    #tickers_to_search: list[str], time_frame: str = "1D"
""" time_frame:
    [1-59]Min or [1-59]T, e.g. 5Min or 5T creates 5-minute aggregations.
    [1-23]Hour or [1-23]H, e.g. 12Hour or 12H creates 12-hour aggregations.
    1Day or 1D creates 1-day aggregations.
    1Week or 1W creates 1-week aggregations.
    [1,2,3,4,6,12]Month or [1,2,3,4,6,12]M, e.g. 3Month or 3M creates 3-month aggregations."""
# wrap these later
is_timeframe_valid, msg = validate_time_frame(time_frame)
is_datetime_valid, msg = validate_datetime(date_start)
is_datetime_valid, msg = validate_datetime(date_end)

sort = "asc" #or desc
next_page_token=""
url = f"https://data.alpaca.markets/v2/stocks/bars? \
    symbols={tickers_to_search} \
    &timeframe={time_frame} \
    &start={date_start} \
    &end={date_end} \
    &limit={limit} \
    &adjustment=raw \
    &feed=sip \
    &page_token={next_page_token} \
    &sort={sort}"
response = requests.get(url, headers=headers)

print(response.text)

In [ ]:
# Date Format: YYYY-MM-DD or YYYY-MM-DDThh:mm:ss where hh is hours in [00,23], mm is minutes in [00, 59], and ss is minutes in [00, 59]. 
#T is delimiter between date and time segments.
# pls start doing things clean and clear ffs. Split this into validate and transform.
def validate_datetime(date_string: str) -> Union[bool, str]:
    logging.basicConfig(filename='datetime.log', level=logging.INFO)
    
    if not isinstance(date_string, str):
        logging.info(print(f"date_string is of type '{type(date_string)}'"))
        return False, "Input must be a string"
    
    # Pattern for YYYY-MM-DD
    yyyy_mm_dd_pattern = r'^\d{4}-\d{2}-\d{2}$'
    # Pattern for YYYY-MM-DDThh:mm:ss (with optional timezone)
    datetime_pattern = r'^\d{4}-\d{2}-\d{2}[Tt ]\d{2}:\d{2}:\d{2}'
    
    date_string = date_string.strip()
    try:
        if re.match(yyyy_mm_dd_pattern, date_string):
            logging.info(print("yyyy_mm_dd matched"))
            dt = datetime.strptime(date_string, "%Y-%m-%d").isoformat()
            logging.info(print(f"Before: {date_string}\nAfter: {dt}\n-------------------------------"))
            return dt, None
        elif re.match(datetime_pattern, date_string):
            logging.info(print("datetime matched"))
            # Replace space with T
            if ' ' in date_string:
                date_string = date_string.replace(' ', 'T')
            # Handle different timezone cases
            if date_string.upper().endswith('Z'):
                # Already has Z timezone
                dt = datetime.fromisoformat(date_string.replace('Z', '+00:00')).isoformat()
                logging.info(print(f"Before: {date_string}\nAfter: {dt}\n-------------------------------"))
                return dt, None
            elif '+' in date_string or '-' in date_string[10:]:
                # Has timezone offset
                dt = datetime.fromisoformat(date_string).isoformat()
                logging.info(print(f"Before: {date_string}\nAfter: {dt}\n-------------------------------"))
                return dt, None
            else:
                # No timezone - assume UTC
                dt = datetime.fromisoformat(date_string).replace(tzinfo=timezone.utc).isoformat()
                logging.info(print(f"Before: {date_string}\nAfter: {dt}\n-------------------------------"))
                return dt, None
        else:
           print("Unsupported date format. Use YYYY-MM-DD or YYYY-MM-DDThh:mm:ss")
            
    except ValueError as e:
        return(f"Invalid date: {str(e)}")
    except Exception as e:
        return(f"Error processing date: {str(e)}")

In [ ]:
date_strings = [
        # Valid YYYY-MM-DD
        "2023-12-25",
        "2000-01-01",
        "1999-12-31",
        
        # Valid RFC-3339
        "2023-12-25T10:30:45Z",
        "2023-12-25T10:30:45+00:00",
        "2023-12-25T10:30:45-05:00",
        "2023-12-25T10:30:45.123Z",
        "2023-12-25T10:30:45.123456+02:00",
        
        # Invalid cases
        "2023-13-25",  # Invalid month
        "2023-12-32",  # Invalid day
        "2023-13-35", # Both day and month are invalid
        "2023/12/25",  # Wrong separator
        "25-12-2023",  # Wrong order
        "2023-12-25T25:30:45Z",  # Invalid hour
        "2023-12-25T10:30:45",  # Missing timezone
        "not-a-date",
        "2023-12-25T10:30:45+5:00",  # Timezone without leading zero
    ]


In [44]:
for date_string in date_strings:
    validate_date_time(date_string)

yyyy_mm_dd matched
Before: 2023-12-25
After: 2023-12-25T00:00:00
-------------------------------
yyyy_mm_dd matched
Before: 2000-01-01
After: 2000-01-01T00:00:00
-------------------------------
yyyy_mm_dd matched
Before: 1999-12-31
After: 1999-12-31T00:00:00
-------------------------------
datetime matched
Before: 2023-12-25T10:30:45Z
After: 2023-12-25 10:30:45+00:00
-------------------------------
datetime matched
Before: 2023-12-25T10:30:45+00:00
After: 2023-12-25 10:30:45+00:00
-------------------------------
datetime matched
Before: 2023-12-25T10:30:45-05:00
After: 2023-12-25 10:30:45-05:00
-------------------------------
datetime matched
Before: 2023-12-25T10:30:45.123Z
After: 2023-12-25 10:30:45.123000+00:00
-------------------------------
datetime matched
Before: 2023-12-25T10:30:45.123456+02:00
After: 2023-12-25 10:30:45.123456+02:00
-------------------------------
yyyy_mm_dd matched
Invalid date: time data '2023-13-25' does not match format '%Y-%m-%d', type e: <class 'ValueErr